### Identify dsDNA and ssDNA viruses

In [ ]:
%%bash
# ### New code:
# micromamba activate dipper
# dipper -i r -o t -I /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_3/uhvdb_r2025_09_dsdna_votu_reps.fna.gz -O dsdna_all_votu_reps_dipper_only.nwk

In [3]:
import polars as pl

# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/uhgv_metadata.tsv
# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/votus_metadata.tsv

uhgv_ictv_taxonomy = (
    pl.read_csv('../figure_1/uhgv_metadata.tsv', separator='\t', columns=['uhgv_genome', 'uhgv_votu'])
        .join(
            pl.read_csv('votus_metadata.tsv', separator='\t', columns=['uhgv_votu', 'ictv_taxonomy', 'genome_length']),
            on='uhgv_votu', how='left'
        )
)


votu_info = (
    pl.read_csv('../figure_2/vclust/uhvdb_vclust_votu_reps_final.tsv', separator='\t', new_columns=['seq_id'])
        .join(
            pl.read_csv('../figure_1/viruses.csvtk_concat.tsv', separator='\t', columns=['seq_name', 'taxonomy', 'contig_length', 'proviral_length']),
            left_on='seq_id', right_on='seq_name', how='left'
        )
        .join(
            pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t'),
            left_on='seq_id', right_on='seq_name', how='left'
        )
        .join(
            uhgv_ictv_taxonomy, left_on='seq_id', right_on='uhgv_genome', how='left'
        )
        .with_columns([
            pl.col('taxonomy').fill_null(pl.col('ictv_taxonomy')),
            pl.when(pl.col('contig_length').is_not_null())
                .then(pl.col('contig_length'))
                .when(pl.col('proviral_length').is_not_null())
                .then(pl.col('proviral_length'))
                .otherwise(pl.col('genome_length')).alias('length').cast(pl.Float64)
        ])
        .unique('seq_id')
)

In [4]:
def extract_realm(value):
    if value is None:
        return ""
    parts = value.split(';')
    if len(parts) > 1:
        if parts[1].strip() == "Anelloviridae":
            return "Monodnaviria"
        if parts[1].strip() == "Naldaviricetes":
            return "Duplodnaviria"
        else:
            return parts[1].strip()
    return ""

print(
    votu_info
        .with_columns([
            pl.col('taxonomy').map_elements(extract_realm, return_dtype=pl.String).alias('ictv_realm'),
        ])
        .group_by('ictv_realm')
        .len()
        .sort('len', descending=True)
)

# # create a list of dsDNA viruses
# (
#     votu_info
#         .with_columns([
#             pl.col('taxonomy').map_elements(extract_realm, return_dtype=pl.String).alias('ictv_realm'),
#         ])
#         .filter(
#             (pl.col('ictv_realm') == 'Duplodnaviria')
#             # (pl.col('ictv_realm') == 'Varidnaviria')
#         )[['seq_id']]
#         .write_csv('dsDNA_viruses.tsv', separator='\t', include_header=False)
# )

# #create a list of ssDNA viruses
# (
#     votu_info
#         .with_columns([
#             pl.col('taxonomy').map_elements(extract_realm, return_dtype=pl.String).alias('ictv_realm'),
#         ])
#         .filter(
#             (pl.col('ictv_realm') == 'Monodnaviria')
#         )[['seq_id']]
#         .write_csv('ssDNA_viruses.tsv', separator='\t', include_header=False)
# )

shape: (5, 2)
┌───────────────┬────────┐
│ ictv_realm    ┆ len    │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ Duplodnaviria ┆ 176413 │
│ Monodnaviria  ┆ 21910  │
│ Varidnaviria  ┆ 2584   │
│               ┆ 528    │
│ Riboviria     ┆ 510    │
└───────────────┴────────┘


### Create tree for ssdna

In [6]:
import polars as pl
import pandas as pd

# identify ssdna genus
ssdna_viruses = set(pl.read_csv('ssDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id'])
family_clusters = pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')

ssdna_families = (
    family_clusters
        .filter(pl.col('contig_id').is_in(ssdna_viruses))
)
print("Number of ssDNA family clusters:", ssdna_families.unique('cluster_id').shape[0])

Number of ssDNA family clusters: 155


In [ ]:
import polars as pl

# ----------------------------
# Step 1: Load data and filter
# ----------------------------
all_norm_scores = pl.read_csv('uhvdb_all.normscores.tsv.gz', separator='\t')

# ssdna_subgenus_centroids = pl.read_csv("ssdna_subgenus_cluster_medoids.tsv", separator="\t")

df = (
    all_norm_scores
        .filter(
            (pl.col('query').is_in(ssdna_viruses)) &
            (pl.col('reference').is_in(ssdna_viruses))
        )
        .with_columns(((100 - pl.col("norm_score")) / 100).alias("dist"))
        .select(["query","reference","dist"])
        .rename({"query":"id1","reference":"id2"})
)

# ----------------------------
# Step 2: Symmetrize distances
# ----------------------------
df_sym = (
    pl.concat([df, df.rename({"id1":"id2","id2":"id1"})[['id1', 'id2', 'dist']]])
    .group_by(["id1","id2"])
    .agg(pl.mean("dist").alias("dist"))
)

# ----------------------------
# Step 3: Build distance dictionary
# ----------------------------
labels = sorted(list(ssdna_viruses))
n = len(labels)
label_to_idx = {label:i for i,label in enumerate(labels)}

dist_dict = {}
for row in df_sym.iter_rows(named=True):
    i = label_to_idx[row["id1"]]
    j = label_to_idx[row["id2"]]
    # always store as (i,j) with i >= j
    if i >= j:
        dist_dict[(i,j)] = row["dist"]
    else:
        dist_dict[(j,i)] = row["dist"]

# ----------------------------
# Step 4: Stream full symmetric PHYLIP
# ----------------------------
with open('figure_3a/ssdna_allvotureps.dist.phylip', 'w') as f:
    f.write(f"{n}\n")
    for i, label_i in enumerate(labels):
        row_values = []
        for j in range(n):
            if i == j:
                dist = 0.0
            elif i > j:
                dist = dist_dict.get((i,j), 1.0)
            else:  # i < j
                dist = dist_dict.get((j,i), 1.0)
            row_values.append(f"{dist:.6f}")  # 6 decimal places
        f.write(f"{label_i} {' '.join(row_values)}\n")


In [ ]:
%%bash

# rapidnj figure_3a/ssdna_allvotureps.dist.phylip \
#     -i pd \
#     -o t \
#     -c 32 \
#     -t p \
#     -x figure_3a/ssdna_allvotureps.nwk

In [ ]:
import polars as pl

# --- Load data ---
# Similarity matrix in long form: genome1, genome2, similarity
similarities = (
    pl.read_csv("../uhvdb_clustering/protein_similarity/uhvdb_family_normscores.tsv", separator="\t", has_header=False, new_columns=["genome1", "genome2", "similarity"])
        .filter(pl.col('genome1').is_in(ssdna_viruses) & pl.col('genome2').is_in(ssdna_viruses))
)

# Cluster assignments: genome, cluster_id
clusters = (
    pl.read_csv("../uhvdb_clustering/protein_similarity/uhvdb_genus_clusters.tsv", separator="\t", has_header=True)
        .filter(pl.col('contig_id').is_in(ssdna_viruses))
)

# --- Join clusters to annotate each genome with its cluster ---
sim_with_clusters = (
    similarities
    .join(clusters.rename({"contig_id": "genome1"}), on="genome1")
    .join(clusters.rename({"contig_id": "genome2", "cluster_id": "cluster_id2"}), on="genome2")
)

# --- Keep only pairs within the same cluster ---
intra_cluster = sim_with_clusters.filter(
    pl.col("cluster_id") == pl.col("cluster_id2")
)

# --- Sum similarity per genome within each cluster ---
sim_sums = (
    intra_cluster
    .group_by(["cluster_id", "genome1"])
    .agg(pl.col("similarity").sum().alias("total_similarity"))
)

# --- Pick medoid = genome with maximum similarity ---
medoids = (
    sim_sums
    .sort(["cluster_id", "total_similarity"], descending=[False, True])
    .group_by("cluster_id")
    .first()
    .rename({"genome1": "medoid"})
)

# --- Add cluster sizes ---
cluster_sizes = clusters.group_by("cluster_id").len().rename({"len": "cluster_size"})

# --- Join medoids with cluster sizes ---
result = medoids.join(cluster_sizes, on="cluster_id")

# --- Add singleton clusters (clusters missing from medoids) ---
# These are clusters with size == 1
singletons = (
    cluster_sizes.filter(pl.col("cluster_size") == 1)
    .join(clusters, on="cluster_id")
    .rename({"contig_id": "medoid"})
    .with_columns(pl.lit(0.0).alias("total_similarity"))
)

# --- Final result = medoids + singletons ---
final_result = pl.concat([result[['cluster_id', 'cluster_size', 'medoid', 'total_similarity']],
    singletons[['cluster_id', 'cluster_size', 'medoid', 'total_similarity']]]).sort("cluster_id")

# --- Save ---
final_result.write_csv("ssdna_genus_cluster_medoids.tsv", separator="\t")


### Annotate ssDNA tree

In [ ]:
# import polars as pl

# # identify ssdna families
# ssdna_viruses = set(pl.read_csv('ssDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id'])
# family_clusters = pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')

# ssdna_families = (
#     family_clusters
#         .filter(pl.col('contig_id').is_in(ssdna_viruses))
# )
# ssdna_families_gt1 = ssdna_families.group_by('cluster_id').len().filter(pl.col('len') >= 1)

# ssdna_seqs_in_families = set(
#     ssdna_families
#         .filter(pl.col('cluster_id').is_in(ssdna_families_gt1['cluster_id']))
#         ['contig_id']
# )

# !mkdir -p ssdna_annotations


# ### Family cluster IDs
# family_clusters = pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
# # 1. calculate median length amd size each cluster
# ssdna_largest_families = set(
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(ssdna_seqs_in_families))
#         )
#         .group_by('cluster_id')
#         .len()
#         .sort('len', descending=True).head(10)['cluster_id']
# )
# print(ssdna_largest_families)

# (
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(ssdna_seqs_in_families))
#         )
#         .with_columns([
#             # assign colors to the largest 10 families {1, 66, 4, 7, 42, 14, 16, 19, 58, 30}
#             pl.when(pl.col('cluster_id') == 4).then(pl.lit('#000000'))
#                 .when(pl.col('cluster_id') == 7).then(pl.lit('#D20103'))
#                 .when(pl.col('cluster_id') == 1).then(pl.lit('#FE9900'))
#                 .when(pl.col('cluster_id') == 19).then(pl.lit('#FFDE59'))
#                 .when(pl.col('cluster_id') == 14).then(pl.lit('#7DDA58'))
#                 .when(pl.col('cluster_id') == 16).then(pl.lit('#5DE2E7'))
#                 .when(pl.col('cluster_id') == 30).then(pl.lit('#060270'))
#                 .when(pl.col('cluster_id') == 42).then(pl.lit('#CC6CE7'))
#                 .when(pl.col('cluster_id') == 58).then(pl.lit('#48015C'))
#                 .when(pl.col('cluster_id') == 66).then(pl.lit('#CECECE'))
#                 .alias('color'),
#             (pl.lit("'") + pl.col('contig_id') + pl.lit("'")).alias('contig_id')
#         ])[['contig_id', 'color', 'cluster_id']]
#         .write_csv('ssdna_annotations/cluster_id_data.tsv', separator='\t', include_header=False)
# )
# !cat colored_strip_dataset_header.txt ssdna_annotations/cluster_id_data.tsv > ssdna_annotations/cluster_id.tsv
# !sed -i 's/DATASET_LABEL\tdb_type/DATASET_LABEL\tcluster_id/g' ssdna_annotations/cluster_id.tsv


# ## ICTV Class ###
# def extract_class(value):
#     if value is None:
#         return ""
#     parts = value.split(';')
#     if len(parts) > 1:
#         for part in parts:
#             if part.endswith('viricetes'):
#                 return part.strip()
#             if part == 'Anelloviridae':
#                 return 'Cardeaviricetes'
#             if part == 'Bamfordvirae':
#                 return 'Bamfordvirae-Class'
#     return ""

# ictv_classes = (
#     votu_info
#         .filter(pl.col('seq_id').is_in(ssdna_seqs_in_families))
#         .with_columns([
#             pl.col('taxonomy').map_elements(extract_class, return_dtype=pl.String).alias('ictv_class')
#         ])
#         [['seq_id', 'ictv_class']]
# )
# print(ictv_classes.group_by('ictv_class').len().sort('len', descending=True))

# (
#     pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
#         .filter(
#             (pl.col('contig_id').is_in(ssdna_seqs_in_families))
#         )
#         .join(ictv_classes, left_on='contig_id', right_on='seq_id', how='left')
#         .with_columns([
#             pl.when(pl.col('ictv_class') == 'Malgrandaviricetes').then(pl.lit('#000000'))
#                 .when(pl.col('ictv_class') == 'Faserviricetes').then(pl.lit('#04023F'))
#                 .when(pl.col('ictv_class') == 'Cardeaviricetes').then(pl.lit('#07028A'))
#                 .when(pl.col('ictv_class') == 'Papovaviricetes').then(pl.lit('#0B02D0'))
#                 .when(pl.col('ictv_class') == 'Arfiviricetes').then(pl.lit('#5A55D7'))
#                 .when(pl.col('ictv_class') == 'Repensiviricetes').then(pl.lit('#5DE2E7'))
#                 .when(pl.col('ictv_class') == 'Quintoviricetes').then(pl.lit('#CECECE'))
#                 .otherwise(pl.lit(''))
#                 .alias('color'),
#             (pl.lit("'") + pl.col('contig_id') + pl.lit("'")).alias('contig_id')
#         ])
#         .filter(pl.col('color') != "")
#         [['contig_id', 'color', 'ictv_class']]
#         .write_csv('ssdna_annotations/ictv_class_data.tsv', separator='\t', include_header=False)
# )
# !cat colored_strip_dataset_header.txt ssdna_annotations/ictv_class_data.tsv > ssdna_annotations/ictv_class.tsv
# !sed -i 's/DATASET_LABEL\tdb_type/DATASET_LABEL\tictv_class/g' ssdna_annotations/ictv_class.tsv


# ### Identify if sequence is new or from prior database
# (
#     pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'db_type']]
#         .filter(pl.col('seq_name').is_in(ssdna_seqs_in_families))
#         .with_columns([
#             pl.when(pl.col('db_type') == 'Database').then(pl.lit('RGBA(0, 0, 0, 50)'))
#                 .otherwise(pl.lit('RGBA(108, 254, 10, 100)'))
#                 .alias('color'),
#             pl.lit('branch').alias('type'),
#             pl.lit('node').alias('what'),
#             pl.lit(1).alias('width_or_size_factor'),
#             pl.lit('normal').alias('style'),
#             (pl.lit("'") + pl.col('seq_name') + pl.lit("'")).alias('seq_name')
#         ])[['seq_name', 'type', 'what', 'color', 'width_or_size_factor', 'style']]
#         .write_csv('ssdna_annotations/db_type_data.tsv', separator='\t', include_header=False)
# )
# !cat style_dataset_header.txt ssdna_annotations/db_type_data.tsv > ssdna_annotations/db_type.tsv


# (
#     pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'body_site']]
#         .filter(pl.col('seq_name').is_in(ssdna_seqs_in_families))
#         .with_columns([
#             pl.when(pl.col('body_site') == 'Oral').then(pl.lit('#0000ff'))
#                 .when(pl.col('body_site') == 'Gut').then(pl.lit('#8D6F64'))
#                 .when(pl.col('body_site') == 'Skin').then(pl.lit('#ff0000'))
#                 .when(pl.col('body_site') == 'Urogenital').then(pl.lit('#FE9900'))
#                 .otherwise(pl.lit('#777777')).alias('color'),
#             (pl.lit("'") + pl.col('seq_name') + pl.lit("'")).alias('seq_name')
#         ])[['seq_name', 'color', 'body_site']]
#         .write_csv('ssdna_annotations/body_site_data.tsv', separator='\t', include_header=False)
# )
# !cat colored_strip_dataset_header.txt ssdna_annotations/body_site_data.tsv > ssdna_annotations/body_site.tsv
# !sed -i 's/DATASET_LABEL\tdb_type/DATASET_LABEL\tbody_site/g' ssdna_annotations/body_site.tsv


# ### Annotate with completeness 
# (
#     pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'checkv_quality']]
#         .filter(pl.col('seq_name').is_in(ssdna_seqs_in_families))
#         .with_columns([
#             pl.when(pl.col('checkv_quality') == 'Complete').then(pl.lit('#4DB624'))
#                 .when(pl.col('checkv_quality') == 'High-quality').then(pl.lit('#FBCE16')).alias('color'),
#             (pl.lit("'") + pl.col('seq_name') + pl.lit("'")).alias('seq_name')
#         ])[['seq_name', 'color', 'checkv_quality']]
#         .write_csv('ssdna_annotations/checkv_quality_data.tsv', separator='\t', include_header=False)
# )
# !cat colored_strip_dataset_header.txt ssdna_annotations/checkv_quality_data.tsv > ssdna_annotations/checkv_quality.tsv
# !sed -i 's/DATASET_LABEL\tdb_type/DATASET_LABEL\tcheckv_quality/g' ssdna_annotations/checkv_quality.tsv

# genome_length = (
#     votu_info
#         .filter(pl.col('seq_id').is_in(ssdna_seqs_in_families))
#         [['seq_id', 'length']]
# )

# (
#     pl.read_csv('../uhvdb_clustering/vclust/uhvdb_vclust_cluster_info_final.tsv', separator='\t')
#         .filter(
#             (pl.col('contig_id').is_in(ssdna_seqs_in_families))
#         )
#         .unique('contig_id')
#         .with_columns((pl.lit("'") + pl.col('contig_id') + pl.lit("'")).alias('contig_id'))
#         .join(genome_length, left_on='contig_id', right_on='seq_id', how='left')[['contig_id', 'length']]
#         .write_csv('ssdna_annotations/genome_length_data.tsv', separator='\t', include_header=False)
# )
# !cat simplebar_dataset_header.txt ssdna_annotations/genome_length_data.tsv > ssdna_annotations/genome_length.tsv


# (
#     votu_info
#         .join(
#             pl.read_csv('genome2taxa.csv'),
#             left_on='seq_id', right_on='votu_rep', how='full'
#         )
#         .filter(
#             (pl.col('votu_rep').is_in(ssdna_seqs_in_families))
#         )
#         .unique('seqhash_rep')
#         .group_by('votu_rep').len()
#         .with_columns([
#             pl.col('len').log10(),
#             (pl.lit("'") + pl.col('votu_rep') + pl.lit("'")).alias('votu_rep')
#         ])
#         .write_csv('ssdna_annotations/votu_genome_count_data.tsv', separator='\t', include_header=False)
# )
# !cat gradient_dataset_header.txt ssdna_annotations/votu_genome_count_data.tsv > ssdna_annotations/votu_genome_count.tsv

{1, 66, 4, 7, 42, 14, 16, 19, 58, 30}


In [12]:
import polars as pl

# identify ssdna families
ssdna_viruses = set(pl.read_csv('ssDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id'])
family_clusters = pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')

### Family cluster IDs
# family_clusters = pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
# # 1. calculate median length amd size each cluster
# ssdna_largest_families = set(
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(ssdna_viruses))
#         )
#         .group_by('cluster_id')
#         .len()
#         .sort('len', descending=True).head(10)['cluster_id']
# )
# print(ssdna_largest_families)

family_clusters = (
    family_clusters
        .filter(
            (pl.col('contig_id').is_in(ssdna_viruses))
        )
        .filter(pl.col('cluster_id').is_in({1, 66, 4, 7, 42, 14, 16, 19, 58, 30}))
        .rename({'contig_id':'seq_name'})
        [['seq_name', 'cluster_id']]
)


## ICTV Class ###
def extract_class(value):
    if value is None:
        return ""
    parts = value.split(';')
    if len(parts) > 1:
        for part in parts:
            if part.endswith('viricetes'):
                return part.strip()
            if part == 'Anelloviridae':
                return 'Cardeaviricetes'
            if part == 'Bamfordvirae':
                return 'Bamfordvirae-Class'
    return ""

ictv_class_info = (
    votu_info
        .filter(pl.col('seq_id').is_in(ssdna_viruses))
        .with_columns([
            pl.col('taxonomy').map_elements(extract_class, return_dtype=pl.String).alias('ictv_class')
        ])
        [['seq_id', 'ictv_class']]
)
print(ictv_class_info.group_by('ictv_class').len().sort('len', descending=True))

ictv_classes = (
    pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
        .join(ictv_class_info, left_on='contig_id', right_on='seq_id', how='left')
        .filter(pl.col('contig_id').is_in(ssdna_viruses))
        .filter(pl.col('ictv_class').is_in(['Malgrandaviricetes', 'Faserviricetes', 'Cardeaviricetes', 'Papovaviricetes', 'Arfiviricetes', 'Repensiviricetes', 'Quintoviricetes']))
        .rename({'contig_id':'seq_name'})
        [['seq_name', 'ictv_class']]
)

### Identify if sequence is new or from prior database
db_type = (
    pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'db_type']]
        .filter(pl.col('seq_name').is_in(ssdna_viruses))
        [['seq_name', 'db_type']]
)


body_site = (
    pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'body_site']]
        .filter(pl.col('seq_name').is_in(ssdna_viruses))
        .with_columns([
            pl.when(pl.col('body_site') == 'Oral').then(pl.lit('Airways'))
                .when(pl.col('body_site') == 'Gut').then(pl.lit('Gut'))
                .when(pl.col('body_site') == 'Skin').then(pl.lit('Skin'))
                .when(pl.col('body_site') == 'Urogenital').then(pl.lit('Urogenital'))
                .otherwise(pl.lit('Other')).alias('body_site'),
        ])[['seq_name', 'body_site']]
)


### Annotate with completeness 
completeness = (
    pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'checkv_quality']]
        .filter(pl.col('seq_name').is_in(ssdna_viruses))
        .with_columns([
            pl.when(pl.col('checkv_quality') == 'Complete').then(pl.lit('Complete'))
                .when(pl.col('checkv_quality') == 'High-quality').then(pl.lit('High-quality')).alias('checkv_quality'),
        ])[['seq_name', 'checkv_quality']]
)

### combine all annotations
all_annotations = (
    family_clusters
        .join(ictv_classes, on='seq_name', how='full', coalesce=True)
        .join(db_type, on='seq_name', how='full', coalesce=True)
        .join(body_site, on='seq_name', how='full', coalesce=True)
        .join(completeness, on='seq_name', how='full', coalesce=True)
        .select([
            'seq_name', 'cluster_id', 'ictv_class', 'db_type', 'body_site', 'checkv_quality'
        ])
)
all_annotations.write_csv('ssdna_annotations/all_ssdna_annotations.tsv', separator='\t')

shape: (8, 2)
┌────────────────────┬───────┐
│ ictv_class         ┆ len   │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ Malgrandaviricetes ┆ 16459 │
│ Faserviricetes     ┆ 3500  │
│ Cardeaviricetes    ┆ 872   │
│ Papovaviricetes    ┆ 559   │
│ Arfiviricetes      ┆ 381   │
│ Repensiviricetes   ┆ 111   │
│                    ┆ 14    │
│ Quintoviricetes    ┆ 14    │
└────────────────────┴───────┘


### Create a tree for dsDNA genus reps

In [2]:
import polars as pl
import pandas as pd

# identify dsdna
dsdna_viruses = set(pl.read_csv('dsDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id'])
family_clusters = pl.read_csv('genome2taxa.csv')

dsdna_families = (
    family_clusters
        .filter(pl.col('votu_rep').is_in(dsdna_viruses))
)
print("Number of dsDNA family clusters:", dsdna_families.unique('family_cluster').shape[0])
print("Number of unique sequences:", dsdna_families.unique('seqhash_rep').shape[0])
print("Number of votu reps:", dsdna_families.unique('votu_rep').shape[0])
print("Number of subgenus reps:", dsdna_families.unique('subgenus_cluster').shape[0])
print("Number of genus reps:", dsdna_families.unique('genus_cluster').shape[0])

dsdna_families_gt20 = dsdna_families.unique('votu_rep').group_by('family_cluster').len().filter(pl.col('len') >= 20)
print("Number of dsDNA family clusters >= 1 members:", dsdna_families_gt20.shape[0])
print("Number of unique dsDNA sequences in families w >= 1 members:", dsdna_families.unique('seqhash_rep').filter(pl.col('family_cluster').is_in(dsdna_families_gt20['family_cluster'])).shape[0])
print("Number of dsDNA votus in families w >= 1 members:", dsdna_families.unique('votu_rep').filter(pl.col('family_cluster').is_in(dsdna_families_gt20['family_cluster'])).shape[0])
print("Number of dsDNA subgenera in families w >= 1 members:", dsdna_families.unique('subgenus_cluster').filter(pl.col('family_cluster').is_in(dsdna_families_gt20['family_cluster'])).shape[0])
print("Number of dsDNA genera in families w >= 1 members:", dsdna_families.unique('genus_cluster').filter(pl.col('family_cluster').is_in(dsdna_families_gt20['family_cluster'])).shape[0])

dsdna_seqs_in_families_gt20 = set(
    dsdna_families
        .filter(pl.col('family_cluster').is_in(dsdna_families_gt20['family_cluster']))
        ['votu_rep']
)

Number of dsDNA family clusters: 2167
Number of unique sequences: 528211
Number of votu reps: 176413
Number of subgenus reps: 95206
Number of genus reps: 42255
Number of dsDNA family clusters >= 1 members: 1021
Number of unique dsDNA sequences in families w >= 1 members: 501366
Number of dsDNA votus in families w >= 1 members: 169131
Number of dsDNA subgenera in families w >= 1 members: 89574
Number of dsDNA genera in families w >= 1 members: 38309


In [1]:
import polars as pl
dsdna_viruses = set(
    pl.read_csv('dsDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id']
)

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [2]:
df = (
    pl.scan_csv('uhvdb_all.normscores.tsv.gz', separator='\t')
        .filter(
            (pl.col('query').is_in(dsdna_viruses)) &
            (pl.col('reference').is_in(dsdna_viruses))
        )
        .sink_csv('figure_3b/dsdna_normscores.tsv', separator='\t', engine='streaming')
)

In [ ]:
!/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_3/TEST/tsv_to_phylip \
    /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_3/figure_3b/dsdna_normscores.tsv \
    /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_3/figure_3b/dsdna_normscores.phylip \
    --similarity

In [ ]:
import polars as pl

# ----------------------------
# Step 1: Load data and filter
# ----------------------------
all_norm_scores = pl.read_csv('uhvdb_all.normscores.tsv.gz', separator='\t')

# dsdna_genus_centroids = pl.read_csv("dsdna_genus_cluster_medoids.tsv", separator="\t")

df = (
    all_norm_scores
        .filter(
            (pl.col('query').is_in(dsdna_seqs_in_families_gt20)) &
            (pl.col('reference').is_in(dsdna_seqs_in_families_gt20))
        )
        .with_columns(((100 - pl.col("norm_score")) / 100).alias("dist"))
        .select(["query","reference","dist"])
        .rename({"query":"id1","reference":"id2"})
)

# ----------------------------
# Step 2: Symmetrize distances
# ----------------------------
df_sym = (
    pl.concat([df, df.rename({"id1":"id2","id2":"id1"})[['id1', 'id2', 'dist']]])
    .group_by(["id1","id2"])
    .agg(pl.mean("dist").alias("dist"))
)

# ----------------------------
# Step 3: Build distance dictionary
# ----------------------------
labels = sorted(list(dsdna_seqs_in_families_gt20))
n = len(labels)
label_to_idx = {label:i for i,label in enumerate(labels)}

dist_dict = {}
for row in df_sym.iter_rows(named=True):
    i = label_to_idx[row["id1"]]
    j = label_to_idx[row["id2"]]
    # always store as (i,j) with i >= j
    if i >= j:
        dist_dict[(i,j)] = row["dist"]
    else:
        dist_dict[(j,i)] = row["dist"]

# ----------------------------
# Step 4: Stream full symmetric PHYLIP
# ----------------------------
with open('dsdna_allvotureps_famgt20.dist.phylip', 'w') as f:
    f.write(f"{n}\n")
    for i, label_i in enumerate(labels):
        row_values = []
        for j in range(n):
            if i == j:
                dist = 0.0
            elif i > j:
                dist = dist_dict.get((i,j), 1.0)
            else:  # i < j
                dist = dist_dict.get((j,i), 1.0)
            row_values.append(f"{dist:.6f}")  # 6 decimal places
        f.write(f"{label_i} {' '.join(row_values)}\n")


In [4]:
import polars as pl

# --- Load data ---
# Similarity matrix in long form: genome1, genome2, similarity
similarities = (
    pl.read_csv("../uhvdb_clustering/protein_similarity/uhvdb_family_normscores.tsv", separator="\t", has_header=False, new_columns=["genome1", "genome2", "similarity"])
        .filter(pl.col('genome1').is_in(dsdna_viruses) & pl.col('genome2').is_in(dsdna_viruses))
)

# Cluster assignments: genome, cluster_id
clusters = (
    pl.read_csv("../uhvdb_clustering/protein_similarity/uhvdb_genus_clusters.tsv", separator="\t", has_header=True)
        .filter(pl.col('contig_id').is_in(dsdna_viruses))
)

# --- Join clusters to annotate each genome with its cluster ---
sim_with_clusters = (
    similarities
    .join(clusters.rename({"contig_id": "genome1"}), on="genome1")
    .join(clusters.rename({"contig_id": "genome2", "cluster_id": "cluster_id2"}), on="genome2")
)

# --- Keep only pairs within the same cluster ---
intra_cluster = sim_with_clusters.filter(
    pl.col("cluster_id") == pl.col("cluster_id2")
)

# --- Sum similarity per genome within each cluster ---
sim_sums = (
    intra_cluster
    .group_by(["cluster_id", "genome1"])
    .agg(pl.col("similarity").sum().alias("total_similarity"))
)

# --- Pick medoid = genome with maximum similarity ---
medoids = (
    sim_sums
    .sort(["cluster_id", "total_similarity"], descending=[False, True])
    .group_by("cluster_id")
    .first()
    .rename({"genome1": "medoid"})
)

# --- Add cluster sizes ---
cluster_sizes = clusters.group_by("cluster_id").len().rename({"len": "cluster_size"})

# --- Join medoids with cluster sizes ---
result = medoids.join(cluster_sizes, on="cluster_id")

# --- Add singleton clusters (clusters missing from medoids) ---
# These are clusters with size == 1
singletons = (
    cluster_sizes.filter(pl.col("cluster_size") == 1)
    .join(clusters, on="cluster_id")
    .rename({"contig_id": "medoid"})
    .with_columns(pl.lit(0.0).alias("total_similarity"))
)

# --- Final result = medoids + singletons ---
final_result = pl.concat([result[['cluster_id', 'cluster_size', 'medoid', 'total_similarity']],
    singletons[['cluster_id', 'cluster_size', 'medoid', 'total_similarity']]]).sort("cluster_id")

# --- Save ---
final_result.write_csv("dsdna_genus_cluster_medoids.tsv", separator="\t")


In [ ]:
%%bash

# rapidnj dsdna_allvotureps_famgt20.dist.phylip \
#     -i pd \
#     -o t \
#     -c 32 \
#     -t p \
#     -x dsdna_allvotureps_famgt20.nwk

### Prune tree to retain one genome per genus

In [5]:
from ete3 import Tree
import polars as pl

# prune tree to retain max of 1,000 genomes per family
# Load your tree (Newick format)
tree = Tree("dsdna_allvotureps_famgt20.nwk", format=1)

# family clusters
genus_medoids = set(
    pl.read_csv("dsdna_genus_cluster_medoids.tsv", separator="\t")['medoid']
)
tips_to_remove = []
for leaf in tree.iter_leaves():
    if leaf.name.replace("'", '') not in genus_medoids:
        tips_to_remove.append(leaf)

# Remove extra genomes
for leaf in tips_to_remove:
    leaf.delete()

tree.write(outfile="dsdna_allvotureps_famgt20_genusmedioids.nwk")

### Annotate dsdna tree

In [9]:
import polars as pl

# identify dsdna families
dsdna_genus_centroids = set(pl.read_csv("dsdna_genus_cluster_medoids.tsv", separator="\t")['medoid'])


!mkdir -p dsdna_annotations

In [ ]:
# import polars as pl

# family_clusters = pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
# # 1. calculate median length amd size each cluster
# dsdna_largest_families = set(
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(dsdna_seqs_in_families))
#         )
#         .group_by('cluster_id')
#         .len()
#         .sort('len', descending=True).head(10)['cluster_id']
# )
# dsdna_largest_families
# (
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(dsdna_seqs_in_families)) &
#             (pl.col('cluster_id').is_in(dsdna_largest_families))
#         )
#         .with_columns([
#             # {2, 3, 5, 6, 8, 9, 10, 11, 12, 13}
#             pl.when(pl.col('cluster_id') == 2).then(pl.lit('A'))
#                 .when(pl.col('cluster_id') == 3).then(pl.lit('B'))
#                 .when(pl.col('cluster_id') == 5).then(pl.lit('C'))
#                 .when(pl.col('cluster_id') == 6).then(pl.lit('D'))
#                 .when(pl.col('cluster_id') == 8).then(pl.lit('E'))
#                 .when(pl.col('cluster_id') == 9).then(pl.lit('F'))
#                 .when(pl.col('cluster_id') == 10).then(pl.lit('G'))
#                 .when(pl.col('cluster_id') == 11).then(pl.lit('H'))
#                 .when(pl.col('cluster_id') == 12).then(pl.lit('I'))
#                 .when(pl.col('cluster_id') == 13).then(pl.lit('J'))
#                 .alias('Cluster_id'),
#         ])
#         [['contig_id', 'cluster_id', 'Cluster_id']]
#         .write_csv('dsdna_annotations/cluster_id_data.tsv', separator='\t', include_header=True)
# )

# ## ICTV Class ###
# def extract_class(value):
#     if value is None:
#         return ""
#     parts = value.split(';')
#     if len(parts) > 1:
#         for part in parts:
#             if part.endswith('viricetes'):
#                 return part.strip()
#             if part == 'Anelloviridae':
#                 return 'Cardeaviricetes'
#             if part == 'Bamfordvirae':
#                 return 'Bamfordvirae-Class'
#     return ""

# ictv_classes = (
#     votu_info
#         .filter(pl.col('seq_id').is_in(dsdna_genus_centroids))
#         .with_columns([
#             pl.col('taxonomy').map_elements(extract_class, return_dtype=pl.String).alias('ictv_class')
#         ])
#         [['seq_id', 'ictv_class']]
# )

# (
#     votu_info
#         .filter(
#             (pl.col('seq_id').is_in(dsdna_genus_centroids))
#         )
#         .join(ictv_classes, on='seq_id', how='left')
#         .with_columns([
#             pl.when(pl.col('ictv_class') == 'Caudoviricetes').then(pl.lit('#04023F'))
#                 .when(pl.col('ictv_class') == 'Bamfordvirae-Class').then(pl.lit('#04023F'))
#                 .when(pl.col('ictv_class') == 'Herviviricetes').then(pl.lit('#07028A'))
#                 .otherwise(pl.lit(''))
#                 .alias('color'),
#         ])
#         .filter(pl.col('color') != "")
#         [['seq_id', 'ictv_class']]
#         .write_csv('dsdna_annotations/ictv_class_data.tsv', separator='\t', include_header=True)
# )


# ### Identify if sequence is new or from prior database
# (
#     pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'db_type']]
#         .filter(pl.col('seq_name').is_in(dsdna_genus_centroids))
#         .with_columns([
#             pl.when(pl.col('db_type') == 'Database').then(pl.lit('#000000'))
#                 .otherwise(pl.lit('#6CFE0A'))
#                 .alias('color'),
#             pl.lit('branch').alias('type'),
#             pl.lit('node').alias('what'),
#             pl.lit(1).alias('width_or_size_factor'),
#             pl.lit('normal').alias('style')
#         ])[['seq_name', 'color']]
#         .write_csv('dsdna_annotations/db_type.tsv', separator='\t', include_header=True)
# )


# (
#     pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'body_site']]
#         .filter(pl.col('seq_name').is_in(dsdna_genus_centroids))
#         .with_columns([
#             pl.when(pl.col('body_site') == 'Oral').then(pl.lit('#0000ff'))
#                 .when(pl.col('body_site') == 'Gut').then(pl.lit('#8D6F64'))
#                 .when(pl.col('body_site') == 'Skin').then(pl.lit('#ff0000'))
#                 .when(pl.col('body_site') == 'Urogenital').then(pl.lit('#FE9900'))
#                 .otherwise(pl.lit('#777777')).alias('color'),
#         ])[['seq_name', 'color', 'body_site']]
#         .write_csv('dsdna_annotations/body_site.tsv', separator='\t', include_header=False)
# )

# ### Annotate with completeness 
# (
#     pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'checkv_quality']]
#         .filter(pl.col('seq_name').is_in(dsdna_genus_centroids))
#         .with_columns([
#             pl.when(pl.col('checkv_quality') == 'Complete').then(pl.lit('#4DB624'))
#                 .when(pl.col('checkv_quality') == 'High-quality').then(pl.lit('#FBCE16')).alias('color'),
#         ])[['seq_name', 'checkv_quality']]
#         .write_csv('dsdna_annotations/checkv_quality.tsv', separator='\t', include_header=True)
# )


# genome_length = (
#     votu_info
#         .filter(pl.col('seq_id').is_in(dsdna_genus_centroids))
#         [['seq_id', 'length']]
# )

# (
#     pl.read_csv('../uhvdb_clustering/vclust/uhvdb_vclust_cluster_info_final.tsv', separator='\t')
#         .filter(
#             (pl.col('contig_id').is_in(dsdna_genus_centroids))
#         )
#         .with_columns([pl.col('length').log10()])
#         .unique('contig_id')
#         .join(genome_length, left_on='contig_id', right_on='seq_id', how='left')[['contig_id', 'length']]
#         .write_csv('dsdna_annotations/genome_length.tsv', separator='\t', include_header=True)
# )


# (
#     votu_info
#         .join(
#             pl.read_csv('genome2taxa.csv'),
#             left_on='seq_id', right_on='votu_rep', how='full'
#         )
#         .filter(
#             (pl.col('votu_rep').is_in(dsdna_genus_centroids))
#         )
#         .unique('seqhash_rep')
#         .group_by('genus_cluster').len()
#         .with_columns([
#             pl.col('len').log10(),
#         ])
#         .write_csv('dsdna_annotations/genus_genome_count.tsv', separator='\t', include_header=True)
# )

In [15]:
import polars as pl

# identify ssdna families
dsdna_viruses = set(pl.read_csv('dsDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id'])
family_clusters = pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')

### Family cluster IDs
family_clusters = pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
# # 1. calculate median length amd size each cluster
# dsdna_largest_families = set(
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(dsdna_viruses))
#         )
#         .group_by('cluster_id')
#         .len()
#         .sort('len', descending=True).head(10)['cluster_id']
# )
# print(dsdna_largest_families)

family_clusters = (
    family_clusters
        .filter(
            (pl.col('contig_id').is_in(dsdna_viruses))
        )
        .filter(pl.col('cluster_id').is_in([2, 3, 5, 6, 8, 9, 10, 11, 12, 13]))
        .rename({'contig_id':'seq_name'})
        [['seq_name', 'cluster_id']]
)


## ICTV Class ###
def extract_class(value):
    if value is None:
        return ""
    parts = value.split(';')
    if len(parts) > 1:
        for part in parts:
            if part.endswith('viricetes'):
                return part.strip()
            if part == 'Anelloviridae':
                return 'Cardeaviricetes'
            if part == 'Bamfordvirae':
                return 'Bamfordvirae-Class'
    return ""

ictv_class_info = (
    votu_info
        .filter(pl.col('seq_id').is_in(dsdna_viruses))
        .with_columns([
            pl.col('taxonomy').map_elements(extract_class, return_dtype=pl.String).alias('ictv_class')
        ])
        [['seq_id', 'ictv_class']]
)
print(ictv_class_info.group_by('ictv_class').len().sort('len', descending=True))

ictv_classes = (
    pl.read_csv('../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
        .join(ictv_class_info, left_on='contig_id', right_on='seq_id', how='left')
        .filter(pl.col('contig_id').is_in(dsdna_viruses))
        .filter(pl.col('ictv_class').is_in(['Caudoviricetes', 'Herviviricetes']))
        .rename({'contig_id':'seq_name'})
        [['seq_name', 'ictv_class']]
)

### Identify if sequence is new or from prior database
db_type = (
    pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'db_type']]
        .filter(pl.col('seq_name').is_in(dsdna_viruses))
        [['seq_name', 'db_type']]
)


body_site = (
    pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'body_site']]
        .filter(pl.col('seq_name').is_in(dsdna_viruses))
        .with_columns([
            pl.when(pl.col('body_site') == 'Oral').then(pl.lit('Airways'))
                .when(pl.col('body_site') == 'Gut').then(pl.lit('Gut'))
                .when(pl.col('body_site') == 'Skin').then(pl.lit('Skin'))
                .when(pl.col('body_site') == 'Urogenital').then(pl.lit('Urogenital'))
                .otherwise(pl.lit('Other')).alias('body_site'),
        ])[['seq_name', 'body_site']]
)


### Annotate with completeness 
completeness = (
    pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'checkv_quality']]
        .filter(pl.col('seq_name').is_in(dsdna_viruses))
        .with_columns([
            pl.when(pl.col('checkv_quality') == 'Complete').then(pl.lit('Complete'))
                .when(pl.col('checkv_quality') == 'High-quality').then(pl.lit('High-quality')).alias('checkv_quality'),
        ])[['seq_name', 'checkv_quality']]
)

### combine all annotations
all_annotations = (
    family_clusters
        .join(ictv_classes, on='seq_name', how='full', coalesce=True)
        .join(db_type, on='seq_name', how='full', coalesce=True)
        .join(body_site, on='seq_name', how='full', coalesce=True)
        .join(completeness, on='seq_name', how='full', coalesce=True)
        .select([
            'seq_name', 'cluster_id', 'ictv_class', 'db_type', 'body_site', 'checkv_quality'
        ])
)
all_annotations.write_csv('dsdna_annotations/all_dsdna_annotations.tsv', separator='\t')

shape: (4, 2)
┌────────────────┬────────┐
│ ictv_class     ┆ len    │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ Caudoviricetes ┆ 176378 │
│                ┆ 19     │
│ Herviviricetes ┆ 15     │
│ Naldaviricetes ┆ 1      │
└────────────────┴────────┘
